## ADP

**ADP** (Aerosol Data Protocol) is a protocol developed by Aerosol d.o.o. for standardized data exchange between instruments and data acquisition systems. It allows for real-time data retrieval, control and monitoring of compatible instruments.

ADP works with **AE36s**, **AE36**, **TCA08** and **TCA09** instruments, while **AE33 is not supported** (AE33 uses legacy AE33 data protocol, which behaves similarly, only commands are different).

**License:** Aerosol Magee Scientific Software License (See LICENSE file for full terms).

In [23]:
from datetime import timedelta, datetime
from io import StringIO

import numpy as np
import pandas as pd

from aerosol_magee_pytools.data_access.tcp_ip import request_tcp, guess_delimiter
from aerosol_magee_pytools.instruments.column_names import COLUMNS_AE36S, COLUMNS_TCA09, COLUMNS_TCA08

DOTNET_EPOCH_OFFSET_SECONDS = 62135596800
DOTNET_EPOCH_OFFSET_MILLISECONDS = 62135596800000
DOTNET_EPOCH_OFFSET_TICKS = 621355968000000000


def dotnet_seconds_to_datetime(series):
    values = pd.Series(pd.to_numeric(series, errors='coerce'), index=series.index)
    valid_values = values.dropna()
    if valid_values.empty:
        return pd.Series(pd.NaT, index=series.index, dtype='datetime64[ns]')

    sample_value = float(valid_values.iloc[0])

    if sample_value >= 1e17:
        unix_values = (values.astype('Int64') - DOTNET_EPOCH_OFFSET_TICKS) * 100
        converted = pd.to_datetime(np.asarray(unix_values, dtype='float64'),
                                   unit='ns',
                                   errors='coerce')
    elif sample_value >= 1e13:
        converted = pd.to_datetime(np.asarray(values, dtype='float64') - DOTNET_EPOCH_OFFSET_MILLISECONDS,
                                   unit='ms',
                                   errors='coerce')
    else:
        converted = pd.to_datetime(np.asarray(values, dtype='float64') - DOTNET_EPOCH_OFFSET_SECONDS,
                                   unit='s',
                                   errors='coerce')

    return pd.Series(converted, index=series.index)

In [24]:
# IP of the instrument - change it to the actual IP address of your AE36s instrument
instrument_ip = '10.10.10.80' # Skylab AE36s
# instrument_ip = '10.10.10.133' # Skylab TCA08
# instrument_ip = '10.10.10.61' # Skylab TCA09

In [25]:
### command INFO
command = '$AERO:INFO\r\n'
received_text = request_tcp(ip=instrument_ip,
                            command=command)
print(received_text)

# parse info response to a dictionary
info = {}
for line in received_text.splitlines():
    line = line.strip()
    if not line or ':' not in line:
        continue
    key, value = line.split(':', 1)
    info[key.strip()] = value.strip()

# there are different fields in INFO response between TCA and AE36, let's synchronize info
if 'Serialnumber' in info:
    info['Serial Number'] = info.pop('Serialnumber' )
    info['Model number'] = info['Serial Number'].split('-')[0]

instrument_typ = info['Model number']
print()
print(f"Instrument type: {instrument_typ}")
print()
print(info)

Connection information
Serialnumber: AE36s-00-00107
Connection ID: 1080
Connection time: 12-Jun-2026 13:41:11
CPU used: 9 % , Memory used: 24 %

Instrument type: AE36s

{'Connection ID': '1080', 'Connection time': '12-Jun-2026 13:41:11', 'CPU used': '9 % , Memory used: 24 %', 'Serial Number': 'AE36s-00-00107', 'Model number': 'AE36s'}


In [26]:
### command LAST return entry from chosen table (table DATA in our case)
command = '$AERO:LAST DATA\r\n'
received_text = request_tcp(ip=instrument_ip,
                            command=command)
print(received_text)

4994471,639168612600000000,639168684600000000,56,1,1,0,0,0,0,1,0,10,10,0,S,868669,369504.625,569145.125,875650,375840.75,604843.625,853281,396457.5,622508.5,854262.75,404048.375,597657.625,852787.5,427233.25,597994.75,868307.625,516268.625,693471.625,872998.75,494550.375,625505.5,867090.125,563674.375,689069.375,865050.375,533148.625,639038,56.04457,18.16614,61.49929,19.88708,57.07838,18.57344,49.58653,15.9819,43.37981,13.82008,37.62799,11.92608,35.10373,11.06756,24.88749,7.711261,23.15945,7.207312,275,327,323,309,355,366,293,321,345,299,300,340,293,251,324,255,222,275,277,216,295,284,288,285,251,188,252,0.002628127,0.002516467,0.0026573,0.002447409,0.002212998,0.001882534,0.001773504,5.557643E-05,0.0001005233,34.1,6.482643,6.754526,5.904438,4.940273,4.254338,3.184036,3.197142,2.212458,1.812378,0.7562809,1.492464,1.037031,0.7977986,0.5101788,-0.1159008,0.106725,0,0,230,101325,25,3791,1204,4995,2478,158,29.6,25.3,28.2,25.8,30.7,33.7,32.8,1376,293,9


**FETCH** command allows you to retrieve data from the instrument for a specific time period.

**Note:** On AE36, AE36s and TCA09, the data is stored in the table named `Data`, while on TCA08, data is stored in the table named `OnlineResults` (table named `Data` on TCA08 is acctually debug_data).

In [27]:
### command FETCH return data from chosen table and time period of last 10 minutes

#### limit time to resent data
end = datetime.now().isoformat(sep=' ', timespec='seconds')
start = (datetime.now() - timedelta(hours=6)).isoformat(sep=' ', timespec='seconds')
###

if instrument_typ != 'TCA08':
    # command = '$AERO:FETCH DATA "2026-06-11 12:51:00" "2026-06-11 15:51:00"\r\n'
    command = f'$AERO:FETCH DATA "{start}" "{end}"\r\n'
else:
    # command = '$AERO:FETCH OnlineResults "2026-06-11 12:51:00" "2026-06-11 15:51:00
    command = f'$AERO:FETCH OnlineResults "{start}" "{end}"\r\n'

received_text = request_tcp(ip=instrument_ip,
                            command=command)
print(received_text)

4994112,639168397200000000,639168469200000000,56,1,1,0,0,0,0,5,0,10,10,0,0,873470.5,457744.625,613583.875,877320.25,471330.625,653872.25,854329.625,489909.875,669114.875,856001.875,488146.375,637048.625,854094.5,504669.375,631906.25,879219.125,603961.625,735689.25,874227.125,566891,653949.125,871480,624735.875,714039.5,869904.875,587408.5,661333.375,35.18098,11.19921,39.05042,12.28343,36.03594,11.47641,30.88183,9.802486,26.87562,8.457314,23.18846,7.265142,21.59254,6.761223,15.10727,4.656627,14.02704,4.337509,895,926,993,1065,1148,1191,1090,1153,1212,1118,1140,1215,1103,1123,1179,1095,1137,1148,1108,1147,1157,1087,1058,1088,1108,1059,1107,0.002820627,0.00270098,0.002791613,0.002587251,0.00240936,0.001982391,0.001980667,0.00006478422,-0.00005735971,9.6,19.96284,21.99971,20.71743,17.66127,15.49549,13.29249,12.54722,8.453618,7.960827,-1.91711,1.89381,2.119467,1.83322,1.189368,0.6837072,0.7389965,0,0,471,101325,25,3805,1190,4995,2466,158,28.8,33.9,26.8,28.4,29.8,32.8,32,1376,292,9
4994113,6

In [28]:
# in the end, you can parse data, for example, convert it into pandas dataframe;
# you will need column names first

if instrument_typ in ['AE36', 'AE36s']:
    columns = COLUMNS_AE36S['Data']
elif instrument_typ == 'TCA08':
    columns = COLUMNS_TCA08['OnlineResult']
elif instrument_typ == 'TCA09':
    columns = COLUMNS_TCA09['Data']
else:
    raise ValueError(f"Unsupported instrument type: {instrument_typ}")

# python can automatically determine the delimiter
separator = guess_delimiter(received_text)

df_data = pd.read_csv(StringIO(received_text.strip()),
                      sep=separator,
                      names=columns)

# convert instrument timestamps stored as .NET epoch seconds to pandas datetime
for column in df_data.columns:
    if ('Timestamp' in column) or column.endswith('TimeUTC') or column.endswith('TimeLocal'):
        df_data[column] = dotnet_seconds_to_datetime(df_data[column])

print(df_data.shape)
print(df_data)

(360, 134)
          ID        TimestampUTC      TimestampLocal  SetupID  G0_Status  \
0    4994112 2026-06-12 05:42:00 2026-06-12 07:42:00       56          1   
1    4994113 2026-06-12 05:43:00 2026-06-12 07:43:00       56          1   
2    4994114 2026-06-12 05:44:00 2026-06-12 07:44:00       56          1   
3    4994115 2026-06-12 05:45:00 2026-06-12 07:45:00       56          1   
4    4994116 2026-06-12 05:46:00 2026-06-12 07:46:00       56          1   
..       ...                 ...                 ...      ...        ...   
355  4994467 2026-06-12 11:37:00 2026-06-12 13:37:00       56          1   
356  4994468 2026-06-12 11:38:00 2026-06-12 13:38:00       56          1   
357  4994469 2026-06-12 11:39:00 2026-06-12 13:39:00       56          1   
358  4994470 2026-06-12 11:40:00 2026-06-12 13:40:00       56          1   
359  4994471 2026-06-12 11:41:00 2026-06-12 13:41:00       56          1   

     G1_Status  G2_Status  G3_Status  G4_Status  G5_Status  G6_Status  \
0  

In [29]:
df_data['TimestampUTC']

0     2026-06-12 05:42:00
1     2026-06-12 05:43:00
2     2026-06-12 05:44:00
3     2026-06-12 05:45:00
4     2026-06-12 05:46:00
              ...        
355   2026-06-12 11:37:00
356   2026-06-12 11:38:00
357   2026-06-12 11:39:00
358   2026-06-12 11:40:00
359   2026-06-12 11:41:00
Name: TimestampUTC, Length: 360, dtype: datetime64[ns]